# 03 — Topic Modeling

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Discover topics with LDA and NMF and check that they line up with the known categories.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### LDA topics
Unsupervised topics learned with no labels.

In [2]:
from src.analysis.topic_modeler import TopicModeler
lda = TopicModeler(n_topics=5, method="lda")
lda.fit_transform(df["text"])
for tid, words in lda.get_all_topics(n_words=8).items():
    print(tid, ":", ", ".join(words))

0 : he, it, but, was, is, his, that, with
1 : it, was, has, is, with, at, by, film
2 : that, is, it, are, be, as, said, will
3 : said, that, is, by, year, has, us, at
4 : he, said, mr, that, be, is, was, it


### NMF topics
A second method as a cross-check. NMF over TF-IDF often gives crisper topics.

In [3]:
nmf = TopicModeler(n_topics=5, method="nmf")
nmf.fit_transform(df["text"])
for tid, words in nmf.get_all_topics(n_words=8).items():
    print(tid, ":", ", ".join(words))

0 : that, is, are, it, be, people, they, will
1 : he, his, we, but, was, game, it, england
2 : mr, he, labour, blair, election, said, party, brown
3 : film, best, her, she, awards, award, was, actor
4 : its, us, said, growth, sales, year, has, it


### Visualize

In [4]:
lda.visualize_topics(n_words=8)

Topic 0: he, it, but, was, is, his, that, with
Topic 1: it, was, has, is, with, at, by, film
Topic 2: that, is, it, are, be, as, said, will
Topic 3: said, that, is, by, year, has, us, at
Topic 4: he, said, mr, that, be, is, was, it


**Takeaway.** Both methods rediscover the five beats (sport, entertainment, tech, business, politics) with no labels, evidence the unsupervised structure is real.